In [ ]:
import random
import datetime
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from seqeval.metrics import f1_score, precision_score, recall_score
import re


CATEGORIES = [
    "Супермаркеты", "Рестораны", "Транспорт", "Связь", "Медицина",
    "Аптеки", "Кафе", "Развлечения", "Шопинг", "Красота",
    "Образование", "Домашние животные", "Дети", "Подарки", "Штрафы",
    "Переводы", "Местный транспорт", "Фастфуд", "Сервис", "Авто",
]


MCC_CODES = ["5411", "5812", "6011", "4814", "5912", "5944", "7997", "3990", "4131", "9999"]


DESCRIPTIONS = [
    "Пятёрочка", "Перекрёсток", "АЗС Лукойл", "Макдоналдс",
    "Яндекс.Такси", "МТС", "Аптека Апрель", "Кофе Хауз",
    "Хлебница", "Wildberries", "Ozon", "Золотое Яблоко",
    "Skillbox", "Четыре Лапы", "Детский Мир", "Flowwow",
    "Штраф ГИБДД", "Магнит", "Табрис",
    "Между своими счетами", "Пополнение Кубышки", "Перевод средств из Кубышки",
    "Перевод на карту", "Яндекс Такси",
]


DATE_FORMATS = ["%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S"]


OPERATION_TYPE_LABELS = {
    "расход": ["Покупка", "Списание", "Оплата", "Платёж", "Платеж"],
    "доход": ["Зачисление", "Возврат", "Поступление"],
    "перевод": ["Перевод", "Перевод на карту", "P2P"],
}


NOISE_PREFIXES = [
    "", "Справка по операции ", "Детали транзакции: ",
    "Выписка ", "Операция по карте ", "Информация: ",
]


NOISE_SUFFIXES = [
    "", " Обработано банком.", " Статус: успешно", " PDF", " Копия.",
]


MID_NOISE = [" MCC ", " банк ", " *** ", " | ", "  "]


SEP_CHOICES = ["\t", " ", " | ", ""]


def _shift_entities(entities, offset):
    for ent in entities:
        ent["start"] += offset
        ent["end"] += offset


def assemble_value_line(pairs, sep_style):
    """Собирает строку значений и span-метки без заголовков."""
    entities = []
    parts = []
    pos = 0
    for i, (key, val) in enumerate(pairs):
        if i > 0:
            sep = random.choice(SEP_CHOICES) if sep_style == "mixed" else sep_style
            parts.append(sep)
            pos += len(sep)
        start = pos
        parts.append(val)
        pos += len(val)
        label = "MCC" if key == "mcc" else key.upper()
        entities.append({"start": start, "end": pos, "label": label})
    return "".join(parts), entities


def apply_noise(text, entities):
    """Добавляет шум до/после/внутри строки, сдвигая span-метки."""
    prefix = random.choice(NOISE_PREFIXES) if random.random() < 0.35 else ""
    suffix = random.choice(NOISE_SUFFIXES) if random.random() < 0.25 else ""

    if prefix:
        _shift_entities(entities, len(prefix))
        text = prefix + text

    if random.random() < 0.15 and len(text) > 10:
        mid = random.choice(MID_NOISE)
        insert_at = random.randint(len(text) // 4, 3 * len(text) // 4)
        text = text[:insert_at] + mid + text[insert_at:]
        for ent in entities:
            if ent["start"] >= insert_at:
                _shift_entities([ent], len(mid))

    return text + suffix, entities


def generate_example_with_headers(pairs, sep="\t"):
    """~15% примеров с заголовками — модель учится игнорировать первую строку."""
    HEADER_SYNONYMS = {
        "operation_date": ["Дата операции", "Дата", "Дата транзакции"],
        "posting_date": ["Дата обработки", "Дата платежа"],
        "time": ["Время", "Время операции"],
        "account": ["Счет", "Карта", "Номер счёта"],
        "amount": ["Сумма", "Сумма операции"],
        "currency": ["Валюта", "Валюта операции"],
        "cashback": ["Кэшбэк", "Спасибо", "Бонусы"],
        "category": ["Категория", "Тип операции"],
        "description": ["Описание", "Детали", "Назначение"],
        "mcc": ["MCC", "МСС"],
        "operation_type": ["Тип", "Операция"],
    }
    headers, values = [], []
    for key, val in pairs:
        headers.append(random.choice(HEADER_SYNONYMS.get(key, [key])))
        values.append(val)
    header_line = sep.join(headers)
    value_line = sep.join(values)
    text = header_line + "\n" + value_line
    value_start = len(header_line) + 1
    entities = []
    pos = 0
    for key, val in pairs:
        if pos > 0:
            pos += len(sep)
        start = value_start + pos
        end = start + len(val)
        label = "MCC" if key == "mcc" else key.upper()
        entities.append({"start": start, "end": end, "label": label})
        pos += len(val)
    return text, entities


def generate_example():
    use_fields = {
        "operation_date": True,
        "posting_date": random.random() < 0.5,
        "time": random.random() < 0.6,
        "account": True,
        "amount": True,
        "currency": random.random() < 0.8,
        "cashback": random.random() < 0.4,
        "category": random.random() < 0.9,
        "description": random.random() < 0.85,
        "operation_type": random.random() < 0.55,
    }

    base_date = datetime.date(2026, 1, 1) + datetime.timedelta(days=random.randint(0, 365))
    op_date_str = base_date.strftime(random.choice(DATE_FORMATS[:3]))
    if random.random() < 0.35:
        t = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
        op_date_str = f"{base_date.strftime('%Y-%m-%d')} {t}"
    posting_date_str = (base_date + datetime.timedelta(days=random.randint(0, 3))).strftime(
        random.choice(["%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"])
    )
    time_str = (
        f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
        if random.random() < 0.5
        else f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}"
    )

    is_expense = random.random() < 0.8
    amount = round(random.uniform(10, 15000), 2)
    if not is_expense:
        amount = -amount if random.random() < 0.25 else amount
    amount_str = f"{amount:,.2f}".replace(",", " ") if random.random() < 0.3 else f"{amount:.2f}"
    amount_str = amount_str.replace(".", random.choice([".", ","]))

    currency = "RUB" if random.random() < 0.9 else random.choice(["USD", "EUR"])
    account = f"*{random.randint(1000, 9999)}"
    category_text = random.choice(CATEGORIES) if random.random() < 0.7 else ""
    mcc = random.choice(MCC_CODES) if random.random() < 0.6 else ""
    description = random.choice(DESCRIPTIONS) if use_fields["description"] else ""
    cashback_str = ""
    if use_fields["cashback"]:
        cashback_val = round(random.uniform(1, 500), 2)
        cashback_str = f"{cashback_val:.2f}".replace(".", random.choice([".", ","]))

    pairs = []
    if use_fields["operation_date"]:
        pairs.append(("operation_date", op_date_str))
    if use_fields["posting_date"]:
        pairs.append(("posting_date", posting_date_str))
    if use_fields["time"]:
        pairs.append(("time", time_str))
    if use_fields["account"]:
        pairs.append(("account", account))
    if use_fields["amount"]:
        pairs.append(("amount", amount_str))
    if use_fields["currency"]:
        pairs.append(("currency", currency))
    if use_fields["cashback"] and cashback_str:
        pairs.append(("cashback", cashback_str))
    if category_text:
        pairs.append(("category", category_text))
    if mcc:
        pairs.append(("mcc", mcc))
    if use_fields["description"] and description:
        pairs.append(("description", description))
    if use_fields["operation_type"]:
        type_key = "расход" if is_expense else random.choice(["доход", "перевод"])
        pairs.append(("operation_type", random.choice(OPERATION_TYPE_LABELS[type_key])))

    random.shuffle(pairs)

    if random.random() < 0.15:
        text, entities = generate_example_with_headers(pairs)
    else:
        sep_style = random.choices(
            ["\t", " ", " | ", "", "mixed"],
            weights=[0.25, 0.25, 0.15, 0.15, 0.20],
            k=1,
        )[0]
        text, entities = assemble_value_line(pairs, sep_style)
        text, entities = apply_noise(text, entities)

    return {"text": text, "entities": entities}


def generate_bank_export_example():
    """Строка в стиле экспорта Т-Банка: дубли дат/сумм, статус OK, нули бонусов."""
    base_date = datetime.date(2026, 1, 1) + datetime.timedelta(days=random.randint(0, 120))
    op_dt = f"{base_date.strftime('%Y-%m-%d')} {random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"
    pay_dt = f"{base_date.strftime('%Y-%m-%d')} 00:00:00"
    account = f"*{random.randint(1000, 9999)}"
    amount = round(random.uniform(50, 15000), 2)
    if random.random() < 0.7:
        amount = -amount
    amount_str = f"{amount:.2f}".rstrip("0").rstrip(".")
    currency = "RUB"
    category = random.choice(CATEGORIES)
    description = random.choice(DESCRIPTIONS) if random.random() < 0.85 else ""
    cashback_str = ""
    if random.random() < 0.25 and amount < 0:
        cashback_str = f"{abs(round(random.uniform(1, 50), 2))}"

    labeled_parts = [
        ("operation_date", op_dt),
        ("posting_date", pay_dt),
        ("account", account),
        ("amount", amount_str),
        ("currency", currency),
    ]
    if random.random() < 0.85:
        labeled_parts.append(("amount", amount_str))
        labeled_parts.append(("currency", currency))
    if cashback_str:
        labeled_parts.append(("cashback", cashback_str))
    labeled_parts.append(("category", category))
    if random.random() < 0.7:
        mcc = random.choice(MCC_CODES)
        labeled_parts.append(("mcc", mcc))
    if description:
        labeled_parts.append(("description", description))

    # шумовые токены без меток
    noise_tokens = ["OK", "0", "0", amount_str] if random.random() < 0.6 else []

    all_tokens = []
    entities = []
    pos = 0
    for item in labeled_parts:
        if isinstance(item, tuple):
            key, val = item
            if all_tokens:
                sep = " "
                all_tokens.append(sep)
                pos += len(sep)
            start = pos
            all_tokens.append(val)
            pos += len(val)
            label = "MCC" if key == "mcc" else key.upper()
            entities.append({"start": start, "end": pos, "label": label})
        else:
            if all_tokens:
                all_tokens.append(" ")
                pos += 1
            all_tokens.append(item)
            pos += len(item)

    for tok in noise_tokens:
        all_tokens.append(" ")
        pos += 1
        all_tokens.append(tok)
        pos += len(tok)

    text = "".join(all_tokens)
    return {"text": text, "entities": entities}


def generate_example_mixed():
    if random.random() < 0.35:
        return generate_bank_export_example()
    return generate_example()


RE_ISO_DT = re.compile(r"^\d{4}-\d{2}-\d{2}(\s+\d{2}:\d{2}(:\d{2})?)?$")
RE_CARD = re.compile(r"^\*\d{4}$")
RE_CURRENCY = re.compile(r"^(RUB|USD|EUR|CNY)$", re.I)
RE_AMOUNT = re.compile(r"^-?\d+([.,]\d+)?$")
RE_MCC = re.compile(r"^\d{4}$")


DESCRIPTION_HINTS = (
    "кубыш", "между своими", "перевод средств", "перевод на",
)


def row_to_raw_text(row):
    """Как при инференсе: все непустые ячейки через пробел."""
    parts = []
    for v in row:
        if pd.notna(v) and str(v).strip().lower() not in ("", "nan"):
            parts.append(str(v).strip())
    return " ".join(parts)


def _looks_like_description(val):
    if val in CATEGORIES:
        return False
    vl = val.lower()
    if any(h in vl for h in DESCRIPTION_HINTS):
        return True
    if re.search(r"[A-Za-z]", val) and len(val) > 2:
        return True
    if " " in val.strip() and len(val) > 10:
        return True
    return False


def _looks_like_category(val):
    return val in CATEGORIES


def _label_cells_for_training(vals):
    labels = [None] * len(vals)
    n_dates = n_amounts = n_currencies = n_cashbacks = 0
    primary_amount = None

    for i, val in enumerate(vals):
        if RE_ISO_DT.match(val):
            labels[i] = "OPERATION_DATE" if n_dates == 0 else ("POSTING_DATE" if n_dates == 1 else None)
            n_dates += 1
        elif RE_CARD.match(val):
            labels[i] = "ACCOUNT"
        elif val.upper() == "OK":
            pass
        elif RE_CURRENCY.match(val):
            if n_currencies == 0:
                labels[i] = "CURRENCY"
            n_currencies += 1
        elif RE_AMOUNT.match(val):
            if n_amounts == 0:
                labels[i] = "AMOUNT"
                try:
                    primary_amount = abs(float(val.replace(",", ".")))
                except ValueError:
                    pass
                n_amounts += 1
            elif (
                n_currencies >= 1
                and val not in ("0", "0.0")
                and n_cashbacks == 0
            ):
                try:
                    fv = abs(float(val.replace(",", ".")))
                    if fv <= 500 and (primary_amount is None or fv <= primary_amount * 0.2):
                        labels[i] = "CASHBACK"
                        n_cashbacks += 1
                except ValueError:
                    pass
            else:
                n_amounts += 1
        elif RE_MCC.match(val) and val.isdigit():
            labels[i] = "MCC"
        elif _looks_like_category(val):
            labels[i] = "CATEGORY"
        elif _looks_like_description(val) or (len(val) > 2 and not RE_AMOUNT.match(val)):
            labels[i] = "DESCRIPTION"

    return labels


def build_ner_example_from_row(row):
    vals = [
        str(v).strip()
        for v in row
        if pd.notna(v) and str(v).strip().lower() not in ("", "nan")
    ]
    text = " ".join(vals)
    cell_labels = _label_cells_for_training(vals)
    entities = []
    search_from = 0
    seen = set()

    for val, lab in zip(vals, cell_labels):
        if not lab:
            continue
        key = (lab, val)
        if lab in ("AMOUNT", "CURRENCY", "POSTING_DATE") and key in seen:
            continue
        seen.add(key)
        idx = text.find(val, search_from)
        if idx < 0:
            idx = text.find(val)
        if idx >= 0:
            entities.append({"start": idx, "end": idx + len(val), "label": lab})
            search_from = idx + len(val)

    return {"text": text, "entities": entities}


def build_ner_dataset_from_excel(excel_path):
    enrich_categories_from_excel(excel_path)
    df = pd.read_excel(excel_path, dtype=str)
    examples = []
    for _, row in df.iterrows():
        ex = build_ner_example_from_row(row)
        if ex["entities"]:
            examples.append(ex)
    print(f"Из Excel: {len(examples)} примеров, категорий в словаре: {len(CATEGORIES)}")
    return examples


def enrich_categories_from_excel(excel_path):
    """1-й проход: частые текстовые метки после блока суммы (только для обучения)."""
    global CATEGORIES
    from collections import Counter

    df = pd.read_excel(excel_path, dtype=str)
    found = set(CATEGORIES)
    candidates = Counter()

    for _, row in df.iterrows():
        vals = [
            str(v).strip()
            for v in row
            if pd.notna(v) and str(v).strip().lower() not in ("", "nan")
        ]
        seen_amount = seen_currency = False
        for val in vals:
            if RE_AMOUNT.match(val) and not seen_amount:
                seen_amount = True
                continue
            if RE_CURRENCY.match(val):
                seen_currency = True
                continue
            if not (seen_amount and seen_currency):
                continue
            if (
                RE_MCC.match(val)
                or RE_CARD.match(val)
                or RE_ISO_DT.match(val)
                or val in ("OK", "RUB", "USD", "EUR")
                or RE_AMOUNT.match(val)
                or re.fullmatch(r"\d+", val)
            ):
                continue
            if _looks_like_description(val):
                continue
            if re.search(r"[А-Яа-яЁё]", val) and len(val) < 40 and len(val.split()) <= 4:
                candidates[val] += 1

    for name, cnt in candidates.items():
        if cnt >= 2 or " " in name or name in found:
            found.add(name)

    CATEGORIES = sorted(found)
    return CATEGORIES

In [ ]:
EXCEL_PATH = r"c:\Users\anusv\Downloads\Telegram Desktop\Operations Fri May 01 2026-Wed May 27 2026.xlsx"

excel_dataset = build_ner_dataset_from_excel(EXCEL_PATH)
synthetic_dataset = [generate_example_mixed() for _ in range(2000)]

# Усилить реальные выписки: каждая строка Excel × 8 + синтетика
dataset = excel_dataset * 8 + synthetic_dataset
pd.DataFrame(dataset).to_csv("receipts_train.csv", index=False)
print(f"Всего примеров: {len(dataset)} (из Excel: {len(excel_dataset)})")

entity_labels = sorted({ent["label"] for ex in dataset for ent in ex["entities"]})
label_list = ["O"] + [f"B-{l}" for l in entity_labels] + [f"I-{l}" for l in entity_labels]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Метки:", label_list)

model_name = "cointegrated/rubert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_offsets(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        return_offsets_mapping=True,
    )
    labels = []
    for i, entities in enumerate(examples["entities"]):
        offsets = tokenized["offset_mapping"][i]
        label_ids = []
        for idx, (start, end) in enumerate(offsets):
            if start == end:
                label_ids.append(-100)
                continue
            token_label = "O"
            for ent in entities:
                if start >= ent["start"] and end <= ent["end"]:
                    prefix = "B" if start == ent["start"] else "I"
                    token_label = f"{prefix}-{ent['label']}"
                    break
            label_ids.append(label2id[token_label])
        labels.append(label_ids)
    tokenized["labels"] = labels
    del tokenized["offset_mapping"]
    return tokenized

dataset_hf = Dataset.from_list(dataset)
tokenized_dataset = dataset_hf.map(tokenize_and_align_offsets, batched=True, remove_columns=dataset_hf.column_names)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

model = BertForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir="./ner_receipts",
    num_train_epochs=15,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=42,
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for _, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    flat_true = [item for sublist in true_labels for item in sublist]
    flat_pred = [item for sublist in true_predictions for item in sublist]
    return {
        "accuracy": accuracy_score(flat_true, flat_pred),
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./ner_receipts_final")
tokenizer.save_pretrained("./ner_receipts_final")
print("Модель обучена.")

In [ ]:
from datetime import datetime as dt
from transformers import AutoModelForTokenClassification
import json


OPERATION_TYPE_KEYWORDS = {
    "перевод": ["перевод", "p2p", "кубышк", "между своими", "своими счет"],
    "доход": ["зачисление", "возврат", "поступление"],
    "расход": ["покупка", "списание", "оплата", "платёж", "платеж"],
}


HEADER_MARKERS = [
    "дата", "сумма", "валюта", "время", "категория",
    "описание", "счет", "счёт", "карта", "кэшбэк", "назначение",
]


OPERATION_TYPE_MAP = {
    'расход': 'expense',
    'доход': 'income',
    'перевод': 'transfer',
}

CURRENCY_MAP = ['RUB', 'USD', 'EUR', 'CNY']

TRANSFER_KEYWORDS = (
    'перевод', 'p2p', 'кубышк', 'между своими', 'своими счет',
)


def excel_to_json(excel_path, debug=False):
    """Excel → сырой текст (склейка ячеек) → NER. Без знания формата/колонок."""
    df = pd.read_excel(excel_path, dtype=str)
    records = []
    for i, row in df.iterrows():
        text = row_to_raw_text(row)
        parsed = parse_receipt(text, debug=debug, model_inf=_model, tokenizer_inf=_tokenizer)
        op_type_key = OPERATION_TYPE_MAP.get(parsed.get('operation_type'))
        currency_raw = (parsed.get('currency') or 'RUB').upper()
        if currency_raw not in CURRENCY_MAP:
            currency_raw = 'OTHER'
        record = {
            'operation_date': parsed.get('operation_date'),
            'amount': parsed.get('amount'),
            'currency': currency_raw,
            'operation_type': op_type_key,
            'account': parsed.get('account') or None,
            'category': parsed.get('category') or None,
            'description': parsed.get('description') or None,
            'cashback': parsed.get('cashback') or None,
            'product_type': None,
            'bank': None,
        }
        if debug:
            print(f"[{i}] {text[:80]}... -> {record}")
        records.append(record)
    return records


def _ent_text(text, ent):
    return text[ent["start"]:ent["end"]]


def _value_line_start(text):
    if "\n" not in text:
        return 0
    header, _ = text.split("\n", 1)
    hits = sum(1 for m in HEADER_MARKERS if m in header.lower())
    return len(header) + 1 if hits >= 2 else 0


def _filter_entities(text, entities):
    vs = _value_line_start(text)
    if vs == 0:
        return entities
    return [e for e in entities if e["start"] >= vs]


def _group_entities(entities):
    grouped = {}
    for ent in entities:
        grouped.setdefault(ent["label"].lower(), []).append(ent)
    return grouped


def _is_decimal_tail(s):
    s = s.strip()
    return bool(re.match(r"^[\.,]\d{1,2}$", s))


def _parse_num(s):
    clean = re.sub(r"[^\d,.\-]", "", s).replace(",", ".")
    try:
        return abs(float(clean))
    except ValueError:
        return 0.0


def _looks_like_card(s):
    s = s.strip()
    return s.startswith("*") or re.fullmatch(r"\*?\d{4}", s)


def _resolve_amount_and_cashback(amount_ents, cashback_ents, text):
    amounts = [_ent_text(text, e) for e in amount_ents]
    cashbacks = [_ent_text(text, e) for e in cashback_ents]

    # убрать номера карт, ошибочно помеченные как сумма/кэшбэк
    amounts = [a for a in amounts if not _looks_like_card(a)]
    cashbacks = [c for c in cashbacks if not _looks_like_card(c)]

    tails = [a for a in amounts if _is_decimal_tail(a)]
    amounts = [a for a in amounts if a not in tails]

    bare_cb_ints = [
        c for c in cashbacks
        if re.fullmatch(r"\d{1,6}", c.strip()) and not re.search(r"[\.,]", c)
    ]
    cashbacks = [c for c in cashbacks if c not in bare_cb_ints]

    # дубли одинаковой суммы — оставить одну
    seen = set()
    dedup_amounts = []
    for a in amounts:
        key = re.sub(r"[^\d.\-]", "", a.replace(",", "."))
        if key not in seen:
            seen.add(key)
            dedup_amounts.append(a)
    amounts = dedup_amounts or amounts

    for tail in tails:
        tail_norm = tail.replace(",", ".")
        if amounts and bare_cb_ints:
            cb = bare_cb_ints.pop(0)
            cashbacks.append(cb.strip() + tail_norm)
        elif bare_cb_ints:
            cb = bare_cb_ints.pop(0)
            if _parse_num(cb) >= 100:
                amounts.append(cb.strip() + tail_norm)
            else:
                cashbacks.append(cb.strip() + tail_norm)
        elif amounts:
            for i, am in enumerate(amounts):
                if re.fullmatch(r"-?\d+", am.strip().replace(" ", "")):
                    amounts[i] = am.strip() + tail_norm
                    break

    cashbacks.extend(bare_cb_ints)

    candidates = [a for a in amounts if not _is_decimal_tail(a)] or amounts
    amount_raw = max(
        candidates,
        key=lambda a: (_parse_num(a), a.lstrip().startswith("-")),
    ) if candidates else "0"

    cashback_raw = max(cashbacks, key=len) if cashbacks else None
    return amount_raw, cashback_raw


def _pick_longest(entities, text, min_len=3):
    if not entities:
        return None
    texts = [_ent_text(text, e) for e in entities]
    good = [t for t in texts if len(t.strip()) >= min_len]
    pool = good or texts
    return max(pool, key=len)


def _pick_account(entities, text):
    if not entities:
        return None
    texts = [_ent_text(text, e) for e in entities]
    starred = [t for t in texts if t.strip().startswith("*")]
    if starred:
        return max(starred, key=len)
    for t in texts:
        digits = re.sub(r"[^\d]", "", t)
        if 3 <= len(digits) <= 4:
            return t
    return _pick_longest(entities, text, min_len=3)


def _pick_category(grouped, text):
    cats = [_ent_text(text, e) for e in grouped.get("category", [])]
    cats = [c for c in cats if not re.fullmatch(r"\d{4}", c.strip())]
    return max(cats, key=len) if cats else ""


def detect_operation_type(text, operation_type_texts, amount_raw):
    for ot in operation_type_texts:
        ot_l = ot.lower()
        for op_type in ("перевод", "доход", "расход"):
            if any(k in ot_l for k in OPERATION_TYPE_KEYWORDS[op_type]):
                return op_type

    text_lower = text.lower()
    for op_type in ("перевод", "доход", "расход"):
        if any(k in text_lower for k in OPERATION_TYPE_KEYWORDS[op_type]):
            return op_type

    if amount_raw and amount_raw.lstrip().startswith("-"):
        return "расход"

    return "доход"


def parse_receipt(text, debug=False, model_inf=None, tokenizer_inf=None):
    if tokenizer_inf is None:
        tokenizer_inf = AutoTokenizer.from_pretrained("./ner_receipts_final")
    if model_inf is None:
        model_inf = AutoModelForTokenClassification.from_pretrained("./ner_receipts_final")
        model_inf.eval()

    inputs = tokenizer_inf(text, return_offsets_mapping=True, return_tensors="pt", truncation=True)
    offsets = inputs.pop("offset_mapping").squeeze(0).tolist()
    with torch.no_grad():
        outputs = model_inf(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2).squeeze(0).tolist()

    entities = []
    current_entity = None
    for pred_id, (start, end) in zip(predictions, offsets):
        if start == end:
            continue
        label = model_inf.config.id2label[pred_id]
        if label == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {"label": label[2:], "start": start, "end": end}
        elif label.startswith("I-") and current_entity and current_entity["label"] == label[2:]:
            current_entity["end"] = end
        else:
            if current_entity:
                entities.append(current_entity)
            etype = label[2:] if label.startswith("I-") else label
            current_entity = {"label": etype, "start": start, "end": end}
    if current_entity:
        entities.append(current_entity)

    entities = _filter_entities(text, entities)

    if debug:
        print("=== Сущности ===")
        for ent in entities:
            print(f"  {ent['label']}: '{_ent_text(text, ent)}'")

    grouped = _group_entities(entities)

    date_str = _pick_longest(grouped.get("operation_date", []), text) or ""
    time_str = _pick_longest(grouped.get("time", []), text) or ""
    op_date = None
    if date_str:
        for fmt in [
            "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M",
            "%d.%m.%Y %H:%M:%S", "%d.%m.%Y %H:%M",
            "%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d",
        ]:
            try:
                op_date = dt.strptime(date_str.strip()[:19], fmt)
                break
            except ValueError:
                continue
        if op_date is None:
            try:
                op_date = pd.to_datetime(date_str).to_pydatetime()
            except Exception:
                pass
    if op_date and time_str:
        for tf in ["%H:%M:%S", "%H:%M"]:
            try:
                t = dt.strptime(time_str, tf).time()
                op_date = dt.combine(op_date.date(), t)
                break
            except ValueError:
                continue

    amount_raw, cashback_raw = _resolve_amount_and_cashback(
        grouped.get("amount", []),
        grouped.get("cashback", []),
        text,
    )
    operation_type_texts = [_ent_text(text, e) for e in grouped.get("operation_type", [])]

    result = {
        "operation_date": op_date.isoformat() if op_date else None,
        "amount": _parse_num(amount_raw),
        "currency": _pick_longest(grouped.get("currency", []), text, min_len=3) or "RUB",
        "operation_type": detect_operation_type(text, operation_type_texts, amount_raw),
        "account": (_pick_account(grouped.get("account", []), text) or "").replace("*", ""),
        "category": _pick_category(grouped, text),
        "description": _pick_longest(grouped.get("description", []), text) or "",
        "cashback": cashback_raw,
    }

    return {k: result.get(k) for k in [
        "operation_date", "amount", "currency", "operation_type",
        "account", "category", "description", "cashback",
    ]}


TEST_CASES = [
    {
        "name": "Табы, минус, расход",
        "text": "26.05.2026\t20:50:54\t-329,97\tRUB\t*0367\tСупермаркеты\tМагнит 3\t12.50",
        "expected_type": "расход",
        "expected_amount": 329.97,
        "expected_cashback": "12.50",
    },
    {
        "name": "Метка Покупка",
        "text": "Покупка 26.05.2026 20:50:54 -329,97 RUB *0367 Супермаркеты Магнит 3",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Зачисление (доход)",
        "text": "Зачисление 15.03.2026 14:22 15000.00 RUB *5958 Поступление зарплаты",
        "expected_type": "доход",
        "expected_amount": 15000.0,
    },
    {
        "name": "Перевод P2P",
        "text": "Перевод на карту 01.02.2026 -5000,00 RUB *1234 Перевод на карту",
        "expected_type": "перевод",
        "expected_amount": 5000.0,
    },
    {
        "name": "Слипшийся текст",
        "text": "26.05.202620:50:54-329,97RUB*0367СупермаркетыМагнит 3",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Шум до и после",
        "text": "Справка по операции 26.05.2026 | 20:50:54 | -329,97 | RUB | *0367 | Супермаркеты | Магнит 3 Обработано банком.",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Положительная сумма без минуса",
        "text": "Оплата 10.01.2026 12:00 1500.00 RUB *7777 Рестораны Макдоналдс",
        "expected_type": "расход",
        "expected_amount": 1500.0,
    },
    {
        "name": "EUR, дата через слэш",
        "text": "20/01/2026 04:30:37 8070.42 EUR *5958 Домашние животные МТС",
        "expected_type": "доход",
        "expected_amount": 8070.42,
    },
    {
        "name": "Старый формат с заголовками (robustness)",
        "text": "Дата операции\tВремя\tСумма\tВалюта\tКарта\tКатегория\tОписание\tКэшбэк\n26.05.2026\t20:50:54\t-329,97\tRUB\t*0367\tСупермаркеты\tМагнит 3\t12.50",
        "expected_type": "расход",
        "expected_amount": 329.97,
        "expected_cashback": "12.50",
    },
    {
        "name": "ISO-дата как в экспорте Т-Банка",
        "text": "2026-05-26 20:50:54 *0367 -329.97 RUB Супермаркеты Магнит 3",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Пополнение Кубышки",
        "text": "2026-05-16 20:16:44 *5919 -1000 RUB Переводы Пополнение Кубышки",
        "expected_type": "перевод",
        "expected_amount": 1000.0,
    },
    {
        "name": "Между своими счетами",
        "text": "2026-05-25 00:36:29 *5919 63.16 RUB Переводы Между своими счетами",
        "expected_type": "перевод",
        "expected_amount": 63.16,
    },
]

_tokenizer = AutoTokenizer.from_pretrained("./ner_receipts_final")
_model = AutoModelForTokenClassification.from_pretrained("./ner_receipts_final")
_model.eval()

EXCEL_PATH = r"c:\Users\anusv\Downloads\Telegram Desktop\Operations Fri May 01 2026-Wed May 27 2026.xlsx"

records = excel_to_json(EXCEL_PATH, debug=False)
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Записей: {len(records)}")
print("Пример:", json.dumps(records[0], ensure_ascii=False, indent=2))
print("Кубышка:", next((r for r in records if r.get('description') and 'убыш' in r['description'].lower()), None))

for case in TEST_CASES:
    print(f"--- {case['name']} ---")
    parsed = parse_receipt(case["text"], debug=True, model_inf=_model, tokenizer_inf=_tokenizer)
    type_ok = parsed["operation_type"] == case["expected_type"]
    amount_ok = abs(parsed["amount"] - case["expected_amount"]) < 0.01 if "expected_amount" in case else True
    cb_ok = True
    if "expected_cashback" in case:
        cb_ok = parsed["cashback"] == case["expected_cashback"]
    print(parsed)
    print(
        f"тип: {'OK' if type_ok else 'FAIL'} ({case['expected_type']} / {parsed['operation_type']}) | "
        f"сумма: {'OK' if amount_ok else 'FAIL'} ({case.get('expected_amount', '-')} / {parsed['amount']}) | "
        f"кэшбэк: {'OK' if cb_ok else 'FAIL'}"
    )
    print()